In [35]:
%pip install scikit-learn pandas numpy xgboost
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

Note: you may need to restart the kernel to use updated packages.


In [3]:
pipes = pd.read_csv("Water_Pipes_Feb10.csv", encoding='cp1252', dtype={10: str})
breaks = pd.read_csv("Breaks2013toNow.csv")
breaks["break_date"] = pd.to_datetime(breaks["break_date"], errors="coerce")
pipes['Soil_Hydro_Group'] = pipes['Soil_Hydro_Group'].fillna("No Group")
pipes = pipes.dropna()
pipes.isna().sum()

OBJECTID            0
tag                 0
status              0
cbu                 0
owner               0
metered             0
fire_line           0
diameter            0
material            0
polywrap            0
yr_inst             0
length              0
yr_abandon          0
yr_removed          0
yr_replace          0
comments            0
NumberOfMa          0
Shape_Leng          0
Score               0
Soil_Comp           0
Soil_Comp_Full      0
Soil_Hydro_Group    0
Shape_Length        0
ElevChange          0
NumberofSLs         0
dtype: int64

In [4]:
def clean_polywrap(x):
    if x == "n":
        return 0.0
    try:
        return float(x)
    except:
        return np.nan


pipes["polywrap"] = pipes["polywrap"].apply(clean_polywrap)
pipes["yr_inst"] = pd.to_numeric(pipes["yr_inst"], errors="coerce")

Snapshots

In [5]:
#print(breaks['break_date'].min(), breaks['break_date'].max())
SNAPSHOT_START = "2013-01-01"
SNAPSHOT_END   = "2025-09-01"

snapshots = pd.date_range(
    SNAPSHOT_START,
    SNAPSHOT_END,
    freq="MS"
)
rows = []

for _, pipe in pipes.iterrows():

    tag = pipe["tag"]
    install_year = pipe["yr_inst"]

    if pd.isna(install_year):
        continue

    pipe_breaks = breaks.loc[
        breaks["pipe_tag"] == tag,
        "break_date"
    ].sort_values()

    for snap in snapshots:

        if snap.year < install_year:
            continue

        past = pipe_breaks[pipe_breaks < snap]

        if len(past) == 0:
            months_since_last = np.nan
            breaks_so_far = 0
        else:
            last_break = past.max()
            months_since_last = (snap - last_break).days / 30.44
            breaks_so_far = len(past)

        rows.append({
            "tag": tag,
            "snapshot": snap,

            "diameter": pipe["diameter"],
            "material": pipe["material"],
            "polywrap": pipe["polywrap"],
            "yr_inst": pipe["yr_inst"],
            "length": pipe["length"],
            "NumberOfMa": pipe["NumberOfMa"],
            "Shape_Length": pipe["Shape_Length"],
            "Score": pipe["Score"],
            "Soil_Comp": pipe["Soil_Comp"],
            "Soil_Hydro_Group": pipe["Soil_Hydro_Group"],
            "ElevChange": pipe["ElevChange"],
            "NumberofSLs": pipe["NumberofSLs"],
            "age_years": snap.year - pipe["yr_inst"],
            "breaks_so_far": breaks_so_far,
            "months_since_last": months_since_last
        })

snap_df = pd.DataFrame(rows)

Preprocesssing

In [6]:
breaks_by_tag = (
    breaks
    .dropna(subset=["pipe_tag", "break_date"])
    .sort_values("break_date")
    .groupby("pipe_tag")["break_date"]
    .apply(np.array)
)

def broke_in_next(tag, snap, months):
    if tag not in breaks_by_tag:
        return 0

    dates = breaks_by_tag[tag]
    start = snap
    end   = snap + pd.DateOffset(months=months)
    start = np.datetime64(start)
    end   = np.datetime64(end)
    i = np.searchsorted(dates, start, side="right")

    if i >= len(dates):
        return 0

    return int(dates[i] <= end)

X = 24 #months
    
snap_df["Y"] = snap_df.apply(
    lambda r: broke_in_next(r["tag"], r["snapshot"], X),
    axis=1
)

XGBoost

In [34]:
feature_cols = [
    #"diameter", #c
    "material", #c
    #"polywrap",
    "yr_inst", #c
    "length",
    "NumberOfMa",
    #"Shape_Length",
    "Score",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    #"age_years",
    "breaks_so_far",  #c
    #"months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [132]:
X_tune, _, y_tune, _ = train_test_split(
    X_train_final,
    y_train,
    train_size=100_000,
    stratify=y_train,
)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'learning_rate': [0.01, 0.001, 0.005, 0.0001], #dont need to test really high (max 0.01 starting from 0.001)
    'max_depth': [3, 5, 10, 15, 999], #most important learning_rate and max_depth also 999
    'subsample': [0.8, 1], #start with 1
    'colsample_bytree': [0.8, 1], # start with 1
    #'gamma': [0, 1, 5],
    'reg_lambda': [0, 5, 10]
}


xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1000,
    tree_method="hist",
)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,
    verbose=1
)

grid_search.fit(X_tune, y_tune)

print(f"Best Score: {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")
#model.get_score()
#for RF, feature selection in scikit-learn (model.feature_importances_)

In [36]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=1500,
    tree_method="hist",
    max_depth = 20,
    #gamma = 1,
    reg_lambda = 10,
    #subsample=0.8,
    #colsample_bytree=0.8,
    learning_rate = 0.1
)
xgb.fit(X_train_final, y_train)
p_test_xgb = xgb.predict_proba(X_test_final)[:, 1]

In [129]:
#xgb.feature_importances_
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = xgb.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
3,Soil,0.553473
1,NumberOfMa,0.247318
6,material,0.147755
4,breaks_so_far,0.016214
2,Score,0.009498
7,yr_inst,0.008935
0,ElevChange,0.008697
5,length,0.008109


In [37]:
print("XGB ROC AUC:", roc_auc_score(y_test, p_test_xgb))
print("XGB PR AUC :", average_precision_score(y_test, p_test_xgb))
print("XGB Log Loss", log_loss(y_test, p_test_xgb))
print("XGB MAE", mean_absolute_error(y_test, p_test_xgb))

XGB ROC AUC: 0.9419106412682854
XGB PR AUC : 0.4135472682814986
XGB Log Loss 0.063410038043463
XGB MAE 0.009067280691984312


random forest

In [21]:
feature_cols = [
    "diameter", #c
    "material", #c
    #"polywrap",
    "yr_inst", #c
    "length",
    "NumberOfMa",
    "Shape_Length",
    "Score",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    "age_years",
    "breaks_so_far",  #c
    "months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

#in case of missing values
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

#one hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
X_train_cat = ohe.fit_transform(X_train[cat_features])
X_test_cat  = ohe.transform(X_test[cat_features])

#train test sets
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [22]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced"
)
rf.fit(X_train_final, y_train)
p_test_rf = rf.predict_proba(X_test_final)[:, 1]

In [23]:
num_names = list(num_features)
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = np.array(num_names + cat_names)
importances = rf.feature_importances_
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})
def base_feature(name):
    if name in num_features:
        return name
    return name.split("_", 1)[0]
fi["base_feature"] = fi["feature"].apply(base_feature)
fi_grouped = (
    fi.groupby("base_feature", as_index=False)["importance"]
      .sum()
      .sort_values("importance", ascending=False)
)

fi_grouped

,base_feature,importance
1,NumberOfMa,0.502046
6,age_years,0.066872
4,Shape_Length,0.063584
9,length,0.059820
3,Score,0.059688
12,yr_inst,0.057286
5,Soil,0.039877
2,NumberofSLs,0.039696
0,ElevChange,0.036459
8,diameter,0.031707


In [ ]:
print("RF ROC AUC:", roc_auc_score(y_test, p_test_rf))
print("RF PR AUC :", average_precision_score(y_test, p_test_rf))
print("RF Log Loss", log_loss(y_test, p_test_rf))
print("RF MAE", mean_absolute_error(y_test, p_test_rf))

RF ROC AUC: 0.9368450563117846
RF PR AUC : 0.40754986549780425
RF Log Loss 0.03847282303628291
RF MAE 0.015395705387062952


GAM testing

In [45]:
from sklearn.preprocessing import OrdinalEncoder

In [52]:
feature_cols = [
    "diameter", #c
    "material", #c
    #"polywrap",
    "yr_inst", #c
    "length",
    "NumberOfMa",
    "Shape_Length",
    "Score",
    "Soil_Comp",
    "Soil_Hydro_Group",
    "ElevChange",
    "NumberofSLs",
    "age_years",
    "breaks_so_far",  #c
    "months_since_last"
]

X = snap_df[feature_cols]

cat_features = [
    "material",
    "Soil_Comp",
    "Soil_Hydro_Group"
]

num_features = [
    c for c in feature_cols if c not in cat_features
]
Xmat = snap_df[num_features + cat_features]
y = snap_df["Y"]

train = snap_df["snapshot"] < "2022-01-01"
X_train = Xmat[train]
X_test  = Xmat[~train]
y_train = y[train]
y_test  = y[~train]

mask = np.ones(len(X_test), dtype=bool)

for col in cat_features:
    mask &= X_test[col].isin(X_train[col].unique())

X_test = X_test.loc[mask].copy()
y_test = y_test.loc[mask].copy()

# numeric
num_imp = SimpleImputer(strategy="median")
X_train_num = num_imp.fit_transform(X_train[num_features])
X_test_num  = num_imp.transform(X_test[num_features])

# categorical -> integer codes
cat_enc = OrdinalEncoder(handle_unknown="error")

X_train_cat = cat_enc.fit_transform(X_train[cat_features])
X_test_cat  = cat_enc.transform(X_test[cat_features])

#train test sets
X_train_cat = cat_enc.fit_transform(X_train[cat_features])
X_test_cat  = cat_enc.transform(X_test[cat_features])

X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num,  X_test_cat])

In [49]:
#%pip install pygam
import pygam
from pygam import LogisticGAM, s, f
from functools import reduce
from operator import add

In [54]:
terms = []

for i in range(len(num_features)):
    terms.append(s(i))

for j in range(len(cat_features)):
    terms.append(f(len(num_features) + j))

terms = reduce(add, terms)

gam = LogisticGAM(terms)

gam.fit(X_train_final, y_train)
p_test_gam = gam.predict_proba(X_test_final)

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\carso\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py", line 3550, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\carso\AppData\Local\Temp\ipykernel_29224\38279847.py", line 13, in <module>
    gam.fit(X_train_final, y_train)
  File "c:\Users\carso\anaconda3\lib\site-packages\pygam\pygam.py", line 912, in fit
    self._pirls(X, y, weights)
  File "c:\Users\carso\anaconda3\lib\site-packages\pygam\pygam.py", line 785, in _pirls
    Q, R = np.linalg.qr(WB.toarray())
  File "<__array_function__ internals>", line 5, in qr
  File "c:\Users\carso\anaconda3\lib\site-packages\numpy\linalg\linalg.py", line 979, in qr
  File "c:\Users\carso\anaconda3\lib\site-packages\numpy\linalg\linalg.py", line 181, in _fastCopyAndTranspose
numpy.core._exceptions._ArrayMemoryError: Unable to allocate 2.65 GiB for an array with shape (1263600, 282) and data type float64

During handling of the 

In [ ]:
print("GAM ROC AUC:", roc_auc_score(y_test, p_test_gam))
print("GAM PR AUC :", average_precision_score(y_test, p_test_gam))
print("GAM Log Loss", log_loss(y_test, p_test_gam))
print("GAM MAE", mean_absolute_error(y_test, p_test_gam))